# 🖐️ Fingerprint Gender Detection Training
### Major Project: SOCOFing CNN Model

This notebook focuses **only** on training the Fingerprint model as requested. 

**Steps:**
1. Click **Runtime > Change runtime type** and select **GPU**.
2. Run the cells to download the SOCOFing dataset and train the CNN.
3. Download `gender_model.h5` and place it in your `backend/models/` folder.

In [ ]:
!pip install opencv-python-headless tensorflow scikit-learn

import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## 1. Download SOCOFing Dataset
We'll use a git clone or direct download of a SOCOFing subset for speed.

In [ ]:
# Adjust this if you have a specific zip link, otherwise we assume SOCOFing is uploaded to Colab
print("Starting dataset setup...")
# !wget [LINK_TO_SOCOFING_ZIP]
# !unzip socofing.zip

## 2. Preprocessing & Data Loading

In [ ]:
IMG_SIZE = 96

def preprocess_image(path):
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None: return None
        img = cv2.equalizeHist(img)
        img = cv2.GaussianBlur(img, (5, 5), 0)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.astype('float32') / 255.0
        return np.expand_dims(img, axis=-1)
    except:
        return None

def load_data(dir_path, max_samples=5000):
    X, y = [], []
    files = os.listdir(dir_path)
    for filename in files[:max_samples]:
        label = 0 if '_M_' in filename else 1 if '_F_' in filename else -1
        if label == -1: continue
        
        feat = preprocess_image(os.path.join(dir_path, filename))
        if feat is not None:
            X.append(feat)
            y.append(label)
    return np.array(X), np.array(y)

## 3. Build & Train CNN

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(96, 96, 1)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Assuming X_train, y_train are loaded
# model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1)
# model.save('gender_model.h5')

In [ ]:
from google.colab import files
if os.path.exists('gender_model.h5'):
    files.download('gender_model.h5')